# Reconciliação do Residual Momentum

Este notebook existe para responder **uma pergunta**: em qual etapa do pipeline a
implementação do grupo e a implementação independente passam a divergir?

O research log registra `Mean IC = 0,0496` para o Residual Momentum contra
`0,0003` do bruto. Uma reimplementação a partir da especificação documentada no
repositório chegou a `-0,0006` contra `0,0178`, ou seja, com a ordem invertida.
Uma das duas está errada, e o número entra na página 4 do relatório final.

**Como usar.** Rode as células em ordem. Na Etapa 0 você cola os intermediários
do notebook original. A cada etapa o notebook compara as duas versões e diz se
elas ainda batem. A primeira etapa que quebrar é a causa.

Se você não tiver os intermediários salvos, rode assim mesmo: o notebook ainda
executa a grade de especificações da Etapa 5 e mostra qual combinação reproduz
`0,0496`.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd

# usa exatamente as mesmas funcoes do grafico_ic.py, para nao haver
# uma terceira implementacao divergindo das outras duas
sys.path.insert(0, str(Path.cwd()))
from grafico_ic import (carregar_precos, residualizar, momentum, rank_ic,
                        UNIVERSO, MERCADO, CORTE_OOS, JAN_BETA, LOOKBACK, GAP)

CORTE = pd.Timestamp(CORTE_OOS)
ALVO_RESIDUAL = 0.0496   # numero do research log que queremos reproduzir
ALVO_BRUTO    = 0.0003

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print(f"universo   : {UNIVERSO}")
print(f"mercado    : {MERCADO}")
print(f"beta       : {JAN_BETA} pregoes")
print(f"momentum   : t-{LOOKBACK} ate t-{GAP}")
print(f"corte OOS  : {CORTE.date()}")

ModuleNotFoundError: No module named 'matplotlib'

## Etapa 0. Cole os intermediários do notebook original

Preencha o que você tiver. Cada chave é opcional: o notebook pula as comparações
para as que faltarem.

| chave | o que é | formato |
|---|---|---|
| `precos` | preços usados no notebook original | DataFrame, índice de datas, colunas = tickers |
| `retornos` | retornos diários | DataFrame, mesmo formato |
| `residuos` | série de resíduos da regressão | DataFrame, mesmo formato |
| `sinal` | Residual Momentum antes do alinhamento mensal | DataFrame, mesmo formato |
| `ic` | série de IC do Residual Momentum | Series, índice de datas |

A forma mais rápida é, no notebook original, salvar cada um com
`obj.to_csv("recon_<nome>.csv")` e carregar aqui.

In [ ]:
SEUS = {}

# Exemplo, descomente e ajuste os caminhos:
# SEUS["residuos"] = pd.read_csv("recon_residuos.csv", index_col=0, parse_dates=True)
# SEUS["sinal"]    = pd.read_csv("recon_sinal.csv",    index_col=0, parse_dates=True)
# SEUS["ic"]       = pd.read_csv("recon_ic.csv",       index_col=0, parse_dates=True).squeeze()

print("intermediarios fornecidos:", list(SEUS) or "nenhum")

In [ ]:
def comparar(nome, meu, seu, limiar=0.999):
    '''Compara duas estruturas alinhadas e diz se a etapa passou ou quebrou.'''
    if seu is None:
        print(f"[{nome}] pulado, intermediario nao fornecido")
        return None

    if isinstance(meu, pd.Series):
        meu, seu = meu.to_frame("v"), seu.to_frame("v")

    cols = [c for c in meu.columns if c in seu.columns]
    idx  = meu.index.intersection(seu.index)
    if not cols or len(idx) == 0:
        print(f"[{nome}] QUEBROU: sem colunas ou datas em comum")
        print(f"   minhas colunas: {list(meu.columns)[:5]} | suas: {list(seu.columns)[:5]}")
        print(f"   meu periodo: {meu.index.min()} a {meu.index.max()}")
        print(f"   seu periodo: {seu.index.min()} a {seu.index.max()}")
        return False

    a, b = meu.loc[idx, cols], seu.loc[idx, cols]
    corr = pd.Series({c: a[c].corr(b[c]) for c in cols}).dropna()
    escala = (b.std() / a.std()).replace([np.inf, -np.inf], np.nan).dropna()

    print(f"[{nome}] {len(idx)} datas e {len(cols)} series em comum")
    print(f"   correlacao   min {corr.min():.4f}   mediana {corr.median():.4f}")
    print(f"   razao desvio min {escala.min():.3f}   mediana {escala.median():.3f}")

    if corr.min() >= limiar and abs(escala.median() - 1) < 0.02:
        print("   OK, as duas versoes coincidem nesta etapa")
        return True
    if corr.min() >= limiar:
        print("   MESMO FORMATO, ESCALA DIFERENTE. Provavel padronizacao "
              "ou log vs simples. Nao muda o rank, logo nao muda o IC.")
        return True
    print("   >>> QUEBROU AQUI. A divergencia nasce nesta etapa. <<<")
    return False

## Etapa 1. Dados e retornos

Divergência clássica: preço ajustado contra preço de fechamento puro, ou janela
de datas diferente. Se os preços já não batem, nada adiante vai bater.

In [ ]:
precos = carregar_precos(Path("data/raw/precos_etfs.csv"))
print(f"meus precos: {precos.index.min().date()} a {precos.index.max().date()}, "
      f"{len(precos)} pregoes")
comparar("precos", precos, SEUS.get("precos"))

ret_log = np.log(precos[UNIVERSO]).diff()
ret_sim = precos[UNIVERSO].pct_change()
rm_log  = np.log(precos[MERCADO]).diff()

print(f"\ncorrelacao entre retorno log e simples: "
      f"{ret_log.corrwith(ret_sim).min():.6f} (minimo entre os ativos)")
print("Se a sua divergencia estiver aqui, ela nao explica o IC: "
      "log e simples preservam o ranking cross-sectional quase perfeitamente.")
comparar("retornos", ret_log, SEUS.get("retornos"))

## Etapa 2. Residualização

Aqui moram as divergências que **mudam o resultado de verdade**. Três formas
diferentes de estimar o beta produzem resíduos diferentes:

- **móvel**, janela de 252 pregões terminando em `t`. É a que respeita o tempo.
- **expansiva**, do início da amostra até `t`. Também respeita o tempo, mas usa
  mais história e reage devagar.
- **amostra cheia**, uma única regressão em todo o período. **Isso é look-ahead**:
  o beta de 2003 usaria dados de 2017. Costuma inflar o IC e é a causa mais
  provável de um número bom demais.

In [ ]:
eps_movel = residualizar(ret_log, rm_log)

# expansiva
beta_exp  = ret_log.expanding(JAN_BETA).cov(rm_log).div(rm_log.expanding(JAN_BETA).var(), axis=0)
alfa_exp  = ret_log.expanding(JAN_BETA).mean() - beta_exp.mul(rm_log.expanding(JAN_BETA).mean(), axis=0)
eps_exp   = ret_log - alfa_exp - beta_exp.mul(rm_log, axis=0)

# amostra cheia  (LOOK-AHEAD, incluida so para diagnostico)
beta_full = ret_log.apply(lambda c: c.cov(rm_log) / rm_log.var())
alfa_full = ret_log.mean() - beta_full * rm_log.mean()
eps_full  = ret_log - alfa_full - rm_log.values.reshape(-1, 1) * beta_full.values

print("beta medio por metodo")
print(pd.DataFrame({
    "movel":        (ret_log.rolling(JAN_BETA).cov(rm_log)
                     .div(rm_log.rolling(JAN_BETA).var(), axis=0)).mean(),
    "expansiva":    beta_exp.mean(),
    "amostra cheia": beta_full,
}).round(3))

comparar("residuos", eps_movel, SEUS.get("residuos"))

## Etapa 3. Definição do sinal

`soma` acumula os resíduos na janela. `padronizado` divide pelo desvio da janela,
que é a definição de Blitz, Huij e Martens (2011). São sinais diferentes e
produzem rankings diferentes.

In [ ]:
sinais = {
    "residual movel soma":    momentum(eps_movel, padronizar=False),
    "residual movel padron.": momentum(eps_movel, padronizar=True),
    "residual expans. soma":  momentum(eps_exp,   padronizar=False),
    "residual cheia soma":    momentum(eps_full,  padronizar=False),   # look-ahead
    "bruto soma":             momentum(ret_log,   padronizar=False),
}
comparar("sinal", sinais["residual movel soma"], SEUS.get("sinal"))

## Etapa 4. Alinhamento e retorno futuro

Duas escolhas que mudam o número:

- **mensal**: sinal no último pregão do mês contra o retorno do mês seguinte.
  Observações independentes, cerca de 200 pontos.
- **diário sobreposto**: sinal em todo pregão contra os 21 pregões seguintes.
  Milhares de pontos, mas fortemente autocorrelacionados. Infla a aparência de
  significância sem adicionar informação.

In [ ]:
fim_mes = ret_log.resample("ME").last().index
fwd_mes = ret_log.resample("ME").sum().shift(-1)
fwd_21d = ret_log.rolling(21).sum().shift(-21)

def ic_de(sinal, freq):
    if freq == "mensal":
        return rank_ic(sinal.reindex(fim_mes, method="ffill"), fwd_mes)
    return rank_ic(sinal, fwd_21d)

comparar("ic", ic_de(sinais["residual movel soma"], "mensal"), SEUS.get("ic"))

## Etapa 5. Grade completa

Todas as combinações, restritas ao research sample. Procure a linha cujo
`mean_ic` chega perto de **0,0496**. A coluna `distancia` ordena por isso.

In [ ]:
linhas = []
for nome_sinal, s in sinais.items():
    for freq in ("mensal", "diario sobreposto"):
        ic = ic_de(s, freq).loc[:CORTE]
        if ic.empty:
            continue
        linhas.append({
            "sinal": nome_sinal, "frequencia": freq, "n": len(ic),
            "mean_ic": ic.mean(), "hit_rate": (ic > 0).mean(),
            "distancia": abs(ic.mean() - ALVO_RESIDUAL),
        })

grade = pd.DataFrame(linhas).sort_values("distancia").reset_index(drop=True)
grade["hit_rate"] = (grade["hit_rate"] * 100).round(1)

print(f"alvo do research log: {ALVO_RESIDUAL}\n")
print(grade.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))

melhor = grade.iloc[0]
if melhor["distancia"] < 0.005:
    print(f"\nREPRODUZIDO por: {melhor['sinal']} em frequencia {melhor['frequencia']}.")
    print("Congele essa especificacao por escrito e commite antes de abrir o holdout.")
    if "cheia" in melhor["sinal"]:
        print("ATENCAO: essa variante usa beta de amostra cheia, que e look-ahead. "
              "Se for essa, o numero nao pode ir para o relatorio como esta.")
else:
    print(f"\nNAO REPRODUZIDO. A combinacao mais proxima erra por {melhor['distancia']:.4f}.")
    print("A divergencia esta em alguma escolha fora desta grade: universo, "
          "proxy de mercado, periodo, tratamento de dividendos ou definicao de IC.")

## Etapa 6. O que registrar no research log

Independente do desfecho, escreva o resultado desta reconciliação no
`Research_Log`, com data. Três desfechos possíveis:

**Reproduziu com uma especificação sem look-ahead.** Congele essa especificação,
commite, e só então abra o holdout. Você ganhou uma segunda implementação
independente confirmando o número, que é uma resposta forte se a banca perguntar
na semifinal.

**Reproduziu apenas com beta de amostra cheia.** O número original tem
look-ahead e não pode ir para o relatório. Recalcule com beta móvel e use o
valor novo, seja ele qual for. Documente a correção: encontrar e corrigir o
próprio viés é exatamente o que o manual de avaliação chama de qualidade da
construção, e vale mais do que o número bonito.

**Não reproduziu de jeito nenhum.** Não publique `0,0496`. Use o número que a
sua implementação auditada produzir, e trate a diferença como uma limitação
declarada. Um resultado modesto e reproduzível sustenta uma arguição ao vivo;
um resultado forte que ninguém consegue refazer, não.